# 11.02 · Models, harness profiles, and token budgets

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2.5 h |
| 📋 Prerequisites | Lesson 11.01; Lesson 01.06 (tokens, cost, rate limits); Lesson 10.03 (intervals). |
| 🔑 Keys | none for 🧪 cells · the key of each model you compare for the 💸 cell (≈ 3–8K tokens per model) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Configure** a deep agent's model with a `provider:model` string, an `init_chat_model` instance, or the course helper `deep_model()` (timeouts, retries, rate limiter).
2. **Shape** the harness per model with **harness profiles** (prompt suffix, hidden tools, the default subagent) and **provider profiles** (init kwargs).
3. **Swap** the model at runtime with a `wrap_model_call` middleware.
4. **Budget** tokens on a free tier: estimate a run before you start it, rate-limit it, cap it, and keep a fallback.
5. **Compare** models on one task (pass rate with an interval, tokens, latency, paid-tier cost) and read the official eval matrix critically.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Models](https://docs.langchain.com/oss/python/deepagents/models) | Suggested models, the eval matrix, `init_chat_model`, provider profiles, runtime model selection |
| [Profiles](https://docs.langchain.com/oss/python/deepagents/profiles) | `HarnessProfile` fields and merge rules (**beta**) |
| [Fault tolerance](https://docs.langchain.com/oss/python/deepagents/fault-tolerance) | Rate limiter, retries, `ModelFallbackMiddleware`, call limits |
| [ChatGoogleGenerativeAI](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai) · [ChatGroq](https://docs.langchain.com/oss/python/integrations/chat/groq) | Provider options |
| [Gemini rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) · [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing) · [Groq rate limits](https://console.groq.com/docs/rate-limits) | Your free-tier budget |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain-google-genai", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-models-budgets'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Strings, instances, and the course helper

`create_deep_agent` accepts a `provider:model` string (resolved with `init_chat_model`) or a chat model instance. The prefix picks the integration: `google_genai:` → `ChatGoogleGenerativeAI` (`langchain-google-genai`, key `GOOGLE_API_KEY`), `groq:` → `ChatGroq`, `openai:` → `ChatOpenAI` (Responses API by default in Deep Agents).

In [ ]:
# 🧪 Two ways to give create_deep_agent a model (no network: constructing a model makes no call).
from langchain.chat_models import init_chat_model

from mastery.deep import deep_model

spec = DEEP_MODEL                                      # 1) a "provider:model" string: create_deep_agent resolves it
configured = init_chat_model("groq:openai/gpt-oss-20b", api_key="offline-placeholder",   # 2) an instance you configure
                             temperature=0, timeout=60, max_retries=2)
course = deep_model("main", api_key="offline-placeholder")   # 3) the course helper: (2) + a rate limiter
print("string     :", spec)
print("instance   :", type(configured).__name__, configured.model_name, "| timeout", configured.request_timeout)
print("course     :", type(course).__name__, "| rate limiter:", type(course.rate_limiter).__name__)
print("profile    :", {k: course.profile.get(k) for k in ("max_input_tokens", "tool_calling")} if course.profile else "n/a")

**What just happened?** Three models, zero calls. The instance carries your settings (`temperature`, `timeout`, `max_retries`). `deep_model()` also attaches an `InMemoryRateLimiter` (at most one call every 4 s by default). `profile` comes from the model's capability data (models.dev): `max_input_tokens` is what summarization uses to decide when the context is "full" (Lesson 11.06).

> ⚠️ **Production note: pin model IDs, never "latest" aliases.** `gemini-3.6-flash` is a stable ID; check the [Gemini deprecations](https://ai.google.dev/gemini-api/docs/deprecations) page before each release. In production: the model ID lives in config, changes go through your eval gate (Lesson 11.11).

## Part 2 · Harness profiles and provider profiles

In [ ]:
# 🧪 A harness profile changes how the deep agent is assembled for ONE provider or model (beta API).
from langchain_core.messages import AIMessage

from deepagents import GeneralPurposeSubagentProfile, HarnessProfile, register_harness_profile
from mastery.deep import build_deep_agent
from mastery.testing import ScriptedChatModel

# Profiles are looked up by "provider:model", then by provider. The scripted model's provider is its class name.
register_harness_profile("scriptedchatmodel", HarnessProfile(
    system_prompt_suffix="Write files in English. Keep every file under 20 lines.",
    excluded_tools=frozenset({"delete", "glob", "grep"}),                     # a smaller, cheaper tool list
    general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),    # no default subagent -> no `task`
))
lean_model = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(lean_model, system_prompt="You are a note taker.", fallbacks=False).invoke(
    {"messages": [{"role": "user", "content": "hi"}]})
print("system prompt the model got:", repr(lean_model.calls[0][0].text))
print("tools the model got        :", lean_model.bound_tools)

**You should now see** your prompt followed by the profile suffix, and only `['ls', 'read_file', 'write_file', 'edit_file']`: the profile hid three tools, and disabling the general-purpose subagent removed `task` (there is nothing left to delegate to). Fewer tools = fewer schema tokens on every call.

Why profiles exist: models differ. Open-weight models often need different tool descriptions or extra repair middleware (the built-in Nemotron profile adds some), and a profile applies those fixes **wherever** that model is used, without touching call sites. `HarnessProfile` fields: `base_system_prompt`, `system_prompt_suffix`, `tool_description_overrides`, `excluded_tools`, `excluded_middleware`, `extra_middleware`, `general_purpose_subagent`.

> ⚠️ **Production note: profiles are beta and global.** `register_harness_profile` changes every agent in the process built with that provider. In production: register profiles once at startup in one module (`mastery/deep/`), test them, and remember they are beta APIs.

In [ ]:
# 🧪 A provider profile sets init kwargs whenever a "provider:model" STRING is resolved (not for instances).
from deepagents import ProviderProfile, register_provider_profile

register_provider_profile("google_genai", ProviderProfile(init_kwargs={"temperature": 0, "max_retries": 3}))
register_provider_profile("groq", ProviderProfile(init_kwargs={"temperature": 0, "max_retries": 3}))
print("registered: every google_genai:* and groq:* string now gets temperature=0, max_retries=3")

**What just happened?** A provider profile only applies when a **string** is resolved. `build_deep_agent` builds instances (to attach the rate limiter), so for course code your settings go to `deep_model(..., temperature=0)` instead. Use provider profiles when you pass strings, for example in `langgraph.json` deployments.

## Part 3 · Swap the model at runtime

In [ ]:
# 🧪 Swap the model at RUNTIME (e.g. a user's dropdown) with wrap_model_call, without rebuilding the agent.
from dataclasses import dataclass

from langchain.agents.middleware import wrap_model_call

cheap = ScriptedChatModel(responses=[AIMessage("answer from the cheap model")], repeat_last=True)
strong = ScriptedChatModel(responses=[AIMessage("answer from the strong model")], repeat_last=True)
CHOICES = {"cheap": cheap, "strong": strong}


@dataclass
class Choice:
    model: str = "cheap"


@wrap_model_call
def chosen_model(request, handler):
    return handler(request.override(model=CHOICES[request.runtime.context.model]))


switchable = build_deep_agent(cheap, middleware=[chosen_model], context_schema=Choice, fallbacks=False)
for pick in ("cheap", "strong"):
    out = switchable.invoke({"messages": [{"role": "user", "content": "hi"}]}, context=Choice(pick))
    print(f"{pick:<6} -> {out['messages'][-1].content}")

**What just happened?** The agent was built once; the `context` of each call picked the model. The same pattern routes easy turns to a cheap model and hard ones to a strong model (Module 04, dynamic model).

## Part 4 · Token budgets on a free tier

In [ ]:
# 🧪 A client-side rate limiter: calls wait their turn instead of hitting a 429.
import time

from langchain_core.rate_limiters import InMemoryRateLimiter

limited = ScriptedChatModel(responses=[AIMessage("x")], repeat_last=True,
                            rate_limiter=InMemoryRateLimiter(requests_per_second=2, check_every_n_seconds=0.05,
                                                             max_bucket_size=1))
start = time.perf_counter()
for _ in range(3):
    limited.invoke("hi")
print(f"3 calls at 2 requests/s took {time.perf_counter() - start:.1f} s (≈ 3 / 2; the bucket starts empty)")

In [ ]:
# 🧪 What a run costs on a free tier: calls × (fixed prompt + growing history). A rough planner.
def plan_tokens(steps: int, fixed: int = 2_000, growth: int = 400, subagents: int = 0, sub_steps: int = 3) -> int:
    """≈ input tokens: each step re-sends `fixed` (system + tool schemas) + all earlier turns (`growth` each)."""
    main = sum(fixed + growth * i for i in range(steps))
    sub = subagents * sum(fixed + growth * i for i in range(sub_steps))
    return main + sub


rows = [{"run": name, "≈ input tokens": f"{plan_tokens(*args):,}"} for name, args in [
    ("small task, 4 steps", (4,)), ("8 steps", (8,)), ("8 steps + 2 subagents", (8, 2_000, 400, 2)),
    ("20 steps + 4 subagents", (20, 2_000, 400, 4, 5))]]
table(rows, ["run", "≈ input tokens"])
print("\nGroq free tier, per model: 8,000 tokens/minute, 200,000 tokens/day (console.groq.com/docs/rate-limits).")
print("Gemini free tier: per project and model, shown in Google AI Studio (ai.google.dev/gemini-api/docs/rate-limits).")

**What just happened?** The planner shows why long runs are expensive: every step re-sends the fixed prompt **and** the growing history, so input tokens grow roughly with the square of the steps, and each subagent pays the fixed prompt again. Your defenses, all on by default in `build_deep_agent`:

| Defense | Stops | Where |
|---|---|---|
| `InMemoryRateLimiter` | bursts that trigger per-minute 429s | `deep_model(requests_per_second=0.25)` |
| `ModelCallLimitMiddleware(run_limit=...)` | runaway loops eating the daily quota | `build_deep_agent(model_calls=25)` |
| `ToolCallLimitMiddleware(run_limit=...)` | tool storms | `build_deep_agent(tool_calls=60)` |
| `ModelFallbackMiddleware` | one model's quota or outage stopping the run | `MASTERY_FALLBACK_MODELS` in `.env` |

**Do this now (5 min):** open [Google AI Studio](https://aistudio.google.com/) → your project → rate limits, and write your live limits for `gemini-3.6-flash` (requests/minute, tokens/minute, requests/day) in `docs/research-log.md`. Google does not publish fixed free-tier numbers, and daily quotas reset at midnight Pacific time.

> ⚠️ **Production note: free tiers are for learning.** On Gemini's unpaid tier, prompts may be used to improve Google's products and read by human reviewers; send only public or synthetic data. In production: a paid tier, per-run budgets, retries with backoff, and a tested fallback model.

## Part 5 · 💸 Compare models on one task

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: one small task on up to three models. Each row ≈ 3–8K tokens; rows without a key are skipped.
import time

from mastery.deep import deep_key_ready, deep_model, run_stats
from mastery.evals.stats import wilson_interval
from mastery.pricing import estimate_cost

RUN_PAID = False      # True also runs the OpenAI benchmark (paid: < $0.01 for this task)
REPEATS = 1           # 3 gives a (wide) interval; it triples the tokens
CANDIDATES = [DEEP_MODEL, "groq:openai/gpt-oss-120b"] + (["openai:gpt-6-luna"] if RUN_PAID else [])
TASK = "Write /notes.md with exactly 3 bullet points on why agents need rate limits. Then reply DONE."


def passed(result) -> bool:
    note = (result.get("files") or {}).get("/notes.md")
    text = note["content"] if isinstance(note, dict) else (note or "")
    return sum(line.lstrip().startswith(("-", "*")) for line in text.splitlines()) == 3


results = []
for spec in dict.fromkeys(CANDIDATES):                 # dict.fromkeys: drop duplicates, keep the order
    if not deep_key_ready(spec):
        print(f"⏭️ {spec}: no key"); continue
    wins, usage, seconds = 0, {"input_tokens": 0, "output_tokens": 0}, 0.0
    try:
        for _ in range(REPEATS):
            agent = build_deep_agent(deep_model(spec=spec), model_calls=8, tool_calls=8, fallbacks=False)
            start = time.perf_counter()
            out = agent.invoke({"messages": [{"role": "user", "content": TASK}]})
            seconds += time.perf_counter() - start
            stats = run_stats(out)
            usage = {k: usage[k] + stats[k] for k in usage}
            wins += passed(out)
    except Exception as exc:                             # one provider failing (429, auth) is a row, not the end
        print(f"❌ {spec}: {type(exc).__name__}: {str(exc)[:120]}"); continue
    cost = estimate_cost(usage, spec)
    results.append({"model": spec, "pass": f"{wins}/{REPEATS}", "95% CI": str(wilson_interval(wins, REPEATS)).split(" (")[0],
                    "in tok": usage["input_tokens"], "out tok": usage["output_tokens"], "s/run": f"{seconds / REPEATS:.1f}",
                    "paid-tier $/run": f"{cost / REPEATS:.5f}" if cost is not None else "n/a"})
if results:
    table(results, ["model", "pass", "95% CI", "in tok", "out tok", "s/run", "paid-tier $/run"])
elif not HAVE_DEEP_KEY:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** one row per model with a key: pass (3 bullets in `/notes.md`), a 95% Wilson interval, tokens, seconds per run, and the paid-tier price of the run. With `REPEATS = 1` the interval is `[0.21, 1.0]` or `[0.0, 0.79]`: **one run tells you almost nothing**. That is why Lesson 11.11 runs every task at least 3 times and reports pass^3.

Read the official [eval matrix](https://docs.langchain.com/oss/python/deepagents/models) the same way: it tests basic operations ("necessary but not sufficient"), Groq-hosted models are not in it, and only your own evals on your tasks decide.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| A provider profile has no effect | You passed a model **instance** | Pass the kwargs to `deep_model()` / `init_chat_model`, or pass a string |
| A harness profile has no effect | Key mismatch (`google_genai` vs `google`) | Use the provider name `get_model_provider` reports (`google_genai`, `groq`, `openai`) |
| `task` tool disappeared | `general_purpose_subagent=...(enabled=False)` and no custom subagents | Expected; add subagents (Lesson 11.05) |
| `429 RESOURCE_EXHAUSTED` (Gemini) / `429 ... (TPD)` (Groq) | Per-minute or per-day quota | Per-minute: wait, lower `requests_per_second`. Per-day: switch model (`MASTERY_FALLBACK_MODELS`) or wait for the reset |
| Thinking tokens make outputs expensive | Gemini bills thinking as output | `deep_model(..., thinking_level="low")` for simple tasks |

## 🏋️ Exercises

1. **(Easy)** Use `plan_tokens` to find the largest number of steps that fits in Groq's 8,000 tokens/minute in **one** minute with the defaults.
2. **(Medium)** Write a harness profile for `groq` that adds `system_prompt_suffix="Always write files before answering."`, and check it with a scripted model whose provider you register (hint: `get_model_provider` from `deepagents._models`).
3. **(Medium)** Extend `chosen_model` so that turns whose last user message is longer than 500 characters use `strong`, and others `cheap`.
4. **(Stretch)** Run Part 5 with `REPEATS = 3` on one model and report pass^3 (all three passed?) next to the pass rate. Which number would you put in a release gate, and why?

<details>
<summary><b>Show solutions</b></summary>

**1.** Three steps: `plan_tokens(2)` = 4,400 fits; `plan_tokens(3)` = 7,200 also fits; `plan_tokens(4)` = 10,400 does not. So **3 steps per minute** with these assumptions: a rate limiter is not optional on Groq's free tier.
```python
print([(s, plan_tokens(s)) for s in range(1, 6)])
```

**2.** Register the scripted model's provider name, since the lookup uses the instance's provider:
```python
from deepagents._models import get_model_provider
m = ScriptedChatModel(responses=[AIMessage("ok")])
register_harness_profile(get_model_provider(m), HarnessProfile(system_prompt_suffix="Always write files before answering."))
```
For a real Groq instance the key is `"groq"`. Remember that this replaces the earlier `scriptedchatmodel` profile from Part 2.

**3.**
```python
@wrap_model_call
def by_length(request, handler):
    last = next((m for m in reversed(request.messages) if m.type == "human"), None)
    model = strong if last and len(last.text) > 500 else cheap
    return handler(request.override(model=model))
```

**4.** pass^3 = 1 only if all three runs pass; it is lower than (or equal to) the pass rate and measures **reliability**, which is what users feel on long runs. Gate on pass^k over a dataset, with the interval's lower bound (Lessons 10.08 and 11.11).
</details>

## 🔎 Docs drill (5 min)

A `HarnessProfile` is registered for `"openai"` and another for `"openai:gpt-5.5"`. Which one applies to `openai:gpt-5.5`, and what happens to the fields the model-level profile does not set?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Profiles](https://docs.langchain.com/oss/python/deepagents/profiles) (or read `??register_harness_profile`).
2. The exact `provider:model` key wins over the provider key.
3. The exact match **inherits** the provider-level defaults for the fields it does not set (merge on top), the same rule as provider profiles on the [Models](https://docs.langchain.com/oss/python/deepagents/models) page.

</details>

## ✅ Checkpoint

- [ ] I pass every deep agent an explicit, configured model, with a rate limiter.
- [ ] I can hide tools or change the prompt for one provider with a harness profile, and I know it is beta and global.
- [ ] I can estimate a run's tokens before starting it, and I logged my live Gemini limits.
- [ ] I compared models with an interval, and I know why one run proves little.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Re-check the suggested-models table and the eval matrix on [Models](https://docs.langchain.com/oss/python/deepagents/models) and the [Gemini deprecations](https://ai.google.dev/gemini-api/docs/deprecations) page. Log in [`docs/research-log.md`](../../docs/research-log.md): is `gemini-3.6-flash` still suggested (2026-10-16: yes, 82% overall)? Does it have a shutdown date? Has a Groq-hosted model been added? Add your live AI Studio limits, with the date.

## Summary

- Strings are resolved with `init_chat_model`; instances carry your settings. The course always passes an instance with a rate limiter.
- Harness profiles reshape the agent per model (prompt, tools, default subagent); provider profiles set init kwargs for strings. Both are beta.
- Free-tier budgets: input grows with steps squared; rate-limit, cap calls, keep a fallback (`MASTERY_FALLBACK_MODELS`).
- Compare models with repeated runs and intervals; the official matrix is necessary, not sufficient.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-03-virtual-filesystem-and-planning.ipynb** → [`11-03-virtual-filesystem-and-planning.ipynb`](11-03-virtual-filesystem-and-planning.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)